### RAG pipelines- Data Ingestion to Vector DB piplines 

In [26]:
import os
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [27]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in directory"""

    all_documents =[]
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print (pdf_files)

    print(f" Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing : {pdf_file.name} ")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:   
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")

    return all_documents

all_pdf_documents = process_all_pdfs("../data")

[WindowsPath('../data/pdf_files/pdf1.pdf'), WindowsPath('../data/pdf_files/pdf2.pdf'), WindowsPath('../data/pdf_files/pdf3.pdf'), WindowsPath('../data/pdf_files/pdf4.pdf')]
 Found 4 PDF files to process
Processing : pdf1.pdf 
  ✓ Loaded 11 pages
Processing : pdf2.pdf 
  ✓ Loaded 11 pages
Processing : pdf3.pdf 
  ✓ Loaded 11 pages
Processing : pdf4.pdf 
  ✓ Loaded 11 pages


In [28]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-04-24T01:06:25+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-24T01:06:25+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf_files\\pdf1.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='Sample PDF Document 1\nThis is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Document 1.\nThis is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Document 1.\nThis is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Document 1.\nThis is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Document 1.\nThis is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Document 1.\nThis is paragraph 1 of Sample PDF Document 1. This is 

In [29]:
### Text splitting get into chunks

def split_documents(documents, chunk_size= 1000, chunk_overlap=200):
    """Split Documents into smaller chunks for better RAG performance."""
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\n Example of Chunks")
        print(f"Content1: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs


In [30]:
chunks = split_documents(all_pdf_documents)

Split 44 documents into 260 chunks

 Example of Chunks
Content1: Sample PDF Document 1
This is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Document 1.
This is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Docume
Metadata: {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-04-24T01:06:25+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-24T01:06:25+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf_files\\pdf1.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}


### Embedding and vector store DB

In [31]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
class EmbeddingManager:
    """Handles document embedding generations using SentenceTransformer"""

    def __init__(self, model_name: str= "all-miniLM-L6-V2"):
        """Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model"""
        try:
            print(f"loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape(len(texts),embedding_dimensions)
        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddinggs for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embedding with shape: {embeddings.shape}")
        return embeddings

## Initialise the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager 


loading embedding model: all-miniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2398.94it/s]


Model loaded successfully. Embedding dimension: 384


### Vectore Store

In [34]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [38]:
### Convert the text into embeddings

texts= [doc.page_content for doc in chunks]

## Generate Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## Store in vector Database
vectorstore.add_documents(chunks,embeddings)


Generating embeddinggs for 260 texts...


Batches: 100%|██████████| 9/9 [00:07<00:00,  1.27it/s]


Generated embedding with shape: (260, 384)
Adding 260 documents to vector store...
Successfully added 260 documents to vector store
Total documents in collection: 260
